# Training pipeline skeleton, Phase 1

I build the skeleton every role uses: I load the clean training tables, I check
the audio folders match, I define the shared config, and I show what the
skeleton gives the team.

Contents:
1. Load the clean training tables
2. Verify the audio folders hold the same counts
3. Define the shared config
4. Look at a sample of the training table
5. Output format for results
6. What this skeleton gives every role

In [1]:
from pathlib import Path

import pandas as pd


def find_repo_root(start):
    p = Path(start).resolve()
    for parent in [p, *p.parents]:
        if (parent / ".git").exists():
            return parent
    raise FileNotFoundError(
        "Repo root not found. Run this notebook from inside the "
        "smart-voicemail-router repo, with the svr-data worktree next to it."
    )


REPO_ROOT = find_repo_root(Path.cwd())
DATA = REPO_ROOT.parent / "svr-data"
TABLES = DATA / "final"
AUDIO = DATA / "dataset"

EXPECTED = {"train": 4031, "val": 1136, "test": 584}


## 1. Load the clean training tables

I load the three tables from the local-env worktree. The tables come from the
SVR Voicemail Corpus: 6,522 voicemails in total, 5,751 clean rows after
preparation. The labels in these tables come from the model.

In [2]:
tables = {}
for split in ["train", "val", "test"]:
    path = TABLES / f"clean_training_{split}.csv"
    tables[split] = pd.read_csv(path)
    n = len(tables[split])
    print(f"{split}: {n} rows from {path.name}")
    assert n == EXPECTED[split], f"{split} has {n} rows, expected {EXPECTED[split]}"

total = sum(len(t) for t in tables.values())
print(f"total clean rows: {total}")
assert total == 5751

train: 4031 rows from clean_training_train.csv
val: 1136 rows from clean_training_val.csv
test: 584 rows from clean_training_test.csv
total clean rows: 5751


## 2. Verify the audio folders hold the same counts

I count the WAV files in each split folder and I compare each count to its
table. The splits must match the audio folders before anyone trains.

In [3]:
for split in ["train", "val", "test"]:
    folder = AUDIO / split
    n_wav = len([p for p in folder.iterdir() if p.suffix.lower() == ".wav"])
    n_rows = len(tables[split])
    print(f"{split}: {n_wav} WAV files, {n_rows} table rows")
    assert n_wav == n_rows, f"{split}: {n_wav} WAV files but {n_rows} rows"

print("Audio folders match the tables on every split.")

train: 4031 WAV files, 4031 table rows
val: 1136 WAV files, 1136 table rows


test: 584 WAV files, 584 table rows
Audio folders match the tables on every split.


## 3. Define the shared config

I define one shared config that every role uses. One config, one seed, one
split, so no one builds on a different setup.

In [4]:
shared_config = {
    "split_seed": 42,
    "experiment_seeds": [42, 1337, 2023, 7, 99],
    "max_seq_len": 128,
    "split_ratio": "70/20/10",
    "dataset_version": "v2",
    "dataset": "svr-voicemail-corpus",
    "train_rows": 4031,
    "val_rows": 1136,
    "test_rows": 584,
}
print(shared_config)


{'split_seed': 42, 'experiment_seeds': [42, 1337, 2023, 7, 99], 'max_seq_len': 128, 'split_ratio': '70/20/10', 'dataset_version': 'v2', 'dataset': 'svr-voicemail-corpus', 'train_rows': 4031, 'val_rows': 1136, 'test_rows': 584}


## 4. Look at a sample of the training table

I look at the first 3 rows of the training table to confirm the four label
fields are there: complaint_category, complaint_urgency, customer_state, and
churn_risk.

In [5]:
cols = ["text_norm", "complaint_category", "complaint_urgency",
        "customer_state", "churn_risk"]
tables["train"].head(3)[cols]

,text_norm,complaint_category,complaint_urgency,customer_state,churn_risk
0,لو سمحتوا ممكن حد يرجع معايا فاتوره الشغله ده ...,billing_and_charges,medium,calm,not_at_risk
1,في باقه مكتوب انها 30 يوم بس كل مره تتجدد بعد ...,balance_and_data,medium,mad,not_at_risk
2,سيد الخير انا حدثك بقالي اسبوع دلوقتي النت اقع...,network_and_connectivity,high,frustrated,not_at_risk


## 5. Output format for results

I define one shape for results so the validation numbers from every role land in the same format. Every result record carries the run id, the model, the split, the seed, and the per-field macro-F1.


In [6]:
def make_result_record(run_id, model, split, seed):
    return {
        "run_id": run_id,
        "model": model,
        "split": split,
        "seed": seed,
        "metrics": {
            "complaint_category_macro_f1": None,
            "complaint_urgency_macro_f1": None,
            "customer_state_macro_f1": None,
            "churn_risk_macro_f1": None,
        },
    }


result_record = make_result_record("001", "EgyBERT", "val", 42)
print(result_record)


{'run_id': '001', 'model': 'EgyBERT', 'split': 'val', 'seed': 42, 'metrics': {'complaint_category_macro_f1': None, 'complaint_urgency_macro_f1': None, 'customer_state_macro_f1': None, 'churn_risk_macro_f1': None}}


## 6. What this skeleton gives every role

This skeleton gives every role four things:

- Data loading: one way to load the clean tables, with the counts checked.
- Shared config: one seed, one split, one dataset version, used by everyone.
- Seed handling: split_seed 42 for the split, and the same 5 experiment_seeds
  for every experiment, so results are comparable.
- Output format: one shape for results, so the validation numbers from every
  role land in the same format.

No one builds on a different setup. That is the point of Phase 1.